# 06 - LangChain com tools

Até aqui o modelo só respondia com texto. Com **tool calling** o modelo pode pedir para executarmos uma função nossa.

O fluxo é:

1. Definimos uma função Python com `@tool` e o modelo recebe a descrição dela.
2. O modelo decide se precisa da tool e devolve os argumentos em `tool_calls`. Ele não executa nada sozinho.
3. Nós executamos a tool com esses argumentos e devolvemos o resultado como `ToolMessage`.
4. O modelo usa o resultado para escrever a resposta final.

Exemplo: o cálculo do IMC é feito pelo nosso código, não pelo modelo.

As configurações de chave e modelo continuam vindo do `.env` na raiz do projeto.

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 1. Definindo a tool

O nome, os argumentos e a docstring da função viram o esquema que o modelo enxerga. A docstring precisa explicar bem para que serve a tool.

In [2]:
@tool
def calcular_imc(peso_kg: float, altura_m: float) -> dict:
    """Calcula o Índice de Massa Corporal (IMC) a partir do peso em quilogramas e da altura em metros."""
    imc = peso_kg / altura_m**2

    if imc < 18.5:
        classificacao = "abaixo do peso"
    elif imc < 25:
        classificacao = "peso normal"
    elif imc < 30:
        classificacao = "sobrepeso"
    else:
        classificacao = "obesidade"

    return {"imc": round(imc, 1), "classificacao": classificacao}


ferramentas = {"calcular_imc": calcular_imc}

## 2. Ligando a tool ao modelo

`bind_tools` envia o esquema da tool junto com cada chamada. Se a resposta tiver `tool_calls`, o modelo está pedindo a execução.

In [3]:
llm_com_tools = llm.bind_tools([calcular_imc])

pergunta_imc = "Meu peso é 80 kg e tenho 1,75 m de altura. Qual é o meu IMC?"

mensagem = llm_com_tools.invoke([HumanMessage(pergunta_imc)])

if mensagem.tool_calls:
    for chamada in mensagem.tool_calls:
        print(f"Tool solicitada pelo modelo: {chamada['name']}")
        print(f"Argumentos: {chamada['args']}")
else:
    print("O modelo respondeu sem chamar nenhuma tool.")

Tool solicitada pelo modelo: calcular_imc
Argumentos: {'peso_kg': 80, 'altura_m': 1.75}


## 3. Executando a tool e devolvendo o resultado

Para cada pedido em `tool_calls`, executamos a função correspondente e devolvemos o resultado como `ToolMessage`, ligado ao `id` da chamada. Depois chamamos o modelo de novo para a resposta final.

In [4]:
mensagens = [HumanMessage(pergunta_imc), mensagem]

for chamada in mensagem.tool_calls:
    resultado = ferramentas[chamada["name"]].invoke(chamada["args"])
    print(f"Executando {chamada['name']}({chamada['args']}) -> {resultado}")
    mensagens.append(
        ToolMessage(content=json.dumps(resultado, ensure_ascii=False), tool_call_id=chamada["id"])
    )

resposta_final = llm_com_tools.invoke(mensagens)

display(Markdown(f"**Pergunta:** {pergunta_imc}\n\n**Resposta:**\n\n{resposta_final.content}"))

Executando calcular_imc({'peso_kg': 80, 'altura_m': 1.75}) -> {'imc': 26.1, 'classificacao': 'sobrepeso'}


**Pergunta:** Meu peso é 80 kg e tenho 1,75 m de altura. Qual é o meu IMC?

**Resposta:**

Seu IMC é 26,1.

Cálculo: IMC = peso / altura² = 80 ÷ (1,75²) = 80 ÷ 3,0625 ≈ 26,1.

Classificação (OMS):
- < 18,5: baixo peso
- 18,5–24,9: peso normal
- 25,0–29,9: sobrepeso
- ≥ 30,0: obesidade (30–34,9 grau I; 35–39,9 grau II; ≥40 grau III)

Interpretação: 26,1 enquadra-se em sobrepeso. Observações: o IMC é uma medida simples e útil em população geral, mas tem limitações (não diferencia músculo de gordura, pode enganar atletas, não é ideal para idosos ou crianças).

Quer que eu calcule um objetivo de peso para ficar na faixa “peso normal”, estime calorias diárias para perder peso com segurança, ou sugira um plano de exercícios/dieta?

## 4. Pergunta que não precisa de tool

Se a pergunta não tiver relação com as tools disponíveis, o modelo responde direto e `tool_calls` fica vazio.

In [5]:
pergunta_geral = "Quem foi Isaac Newton?"

mensagem_geral = llm_com_tools.invoke([HumanMessage(pergunta_geral)])

display(Markdown(f"**Pergunta:** {pergunta_geral}\n\n**Tool calls:** {mensagem_geral.tool_calls}\n\n**Resposta:**\n\n{mensagem_geral.content}"))

**Pergunta:** Quem foi Isaac Newton?

**Tool calls:** []

**Resposta:**

Isaac Newton (nascido em Inglaterra em 25 de dezembro de 1642 pelo calendário juliano — 4 de janeiro de 1643 no calendário gregoriano; faleceu em 20 de março de 1726 juliano — 31 de março de 1727 gregoriano) foi um dos mais influentes cientistas da história. Foi matemático, físico, astrônomo, alquimista e teólogo.  

Principais realizações e cargos
- Autor de Philosophiæ Naturalis Principia Mathematica (1687), obra que fundou a mecânica clássica rigorosa.  
- Formulou as três leis do movimento (base da dinâmica) e a lei da gravitação universal (força gravitacional entre massas decai com o quadrado da distância).  
- Co‑criador do cálculo diferencial e integral (sua versão era chamada de método das fluxões); houve controvérsia de prioridade com Gottfried Wilhelm Leibniz.  
- Fez importantes descobertas em óptica: mostrou que a luz branca se decompõe em cores por refração em prismas, desenvolveu o telescópio refletor.  
- Ocupou cargos públicos: Professor Lucasiano em Cambridge, Master of the Mint (responsável pela Casa da Moeda britânica) e presidente da Royal Society; foi cavaleiro (knighted) em 1705.

Impacto e outras notas
- Sua formulação das leis do movimento e da gravitação dominou a física até a revolução da relatividade e da mecânica quântica, e ainda é usada para resolver a maioria dos problemas de engenharia e astronomia prática.  
- Escreveu também sobre teologia e alquimia; muitos desses escritos eram menos compatíveis com o espírito científico moderno e só vieram a público mais tarde.  
- A vida científica de Newton foi marcada tanto por grandes avanços teóricos quanto por disputas (por exemplo, com Leibniz) e por um estilo de trabalho bastante reservado.

Em resumo: Newton é considerado um dos fundadores da ciência moderna por ter fornecido leis e métodos matemáticos que explicam e permitem prever o movimento dos corpos e muitos fenômenos naturais. Quer que eu aprofunde algum desses pontos (Princípios, leis do movimento, cálculo, óptica ou a disputa com Leibniz)?

## Observações

- O modelo só escolhe a tool e preenche os argumentos. Quem executa é o nosso código, então a tool pode fazer o que for preciso, como consultar banco ou API.
- Os argumentos vêm do texto do usuário e podem estar errados ou faltando. Em tools de verdade, valide os valores antes de usar.
- Este exemplo trata uma chamada por vez. Se o modelo pedir várias tools numa resposta, o laço do passo 3 já executa todas.
- Em fluxos com vários passos, o mesmo ciclo pode ser montado com LangGraph, como no notebook `02-Langgraph`.